# Prüfungsleistung - Strukturierte Informationsextraktion aus Stellenanzeigen

**Name:** *Jakob Wilmesmeier*

---

## Kontext: Die JobMatch-Plattform

Die (fiktive) HR-Tech-Firma **JobMatch** betreibt eine Karriereplattform, auf der Bewerber:innen passende Stellenanzeigen finden. Täglich werden tausende Anzeigen aus dem Web gecrawlt, aber als unstrukturierter Volltext sind sie für die Nutzer:innen kaum filterbar.

JobMatch braucht ein Modell, das aus jeder Anzeige genau die Felder extrahiert, nach denen Bewerber:innen tatsächlich filtern: Jobtitel, Unternehmen, Erfahrungslevel, Skills, Bildungsanforderungen, Gehaltsspanne und Benefits.

## Aufgabe

Entwickle ein Modell, das aus einer englischsprachigen Stellenanzeige (Freitext) ein strukturiertes JSON-Objekt mit sieben vordefinierten Feldern extrahiert:

```json
{
  "job_title":              "Senior Data Engineer",
  "company":                "Acme Corp",
  "experience_level":       "senior",
  "required_skills":        ["Python", "SQL", "Airflow"],
  "education_requirements": "Bachelor's degree in Computer Science",
  "salary_range":           {"min": 80000, "max": 110000, "currency": "USD"},
  "compensation_benefits":  ["health insurance", "401k matching", "remote work"]
}
```

Feldtypen im Detail:
- `job_title` (string): Bezeichnung der Stelle
- `company` (string): Name des Unternehmens
- `experience_level` (enum): Einer von `entry`, `junior`, `mid`, `senior`, `lead`, `executive`, `unknown`
- `required_skills` (list[string]): Geforderte technische oder fachliche Skills
- `education_requirements` (string): Bildungsanforderungen (oder leerer String)
- `salary_range` (object): `{min, max, currency}`, jeweils `null` wenn nicht angegeben
- `compensation_benefits` (list[string]): Benefits (Versicherungen, Urlaub, Remote etc.)

Felder können leer sein, wenn die Anzeige keine entsprechende Information enthält (leerer String, leere Liste, `null`-Subfelder bei `salary_range`).

**Inferenz-Laufzeit**: Das Ausführen des Auswertungs-Abschnitts (Modell laden + `predict()` auf 50 Testanzeigen + `evaluate()`) darf insgesamt maximal 20 Minuten dauern, gemessen auf einer Colab-T4-GPU. Plane dein Modell entsprechend. Bei Überschreitung gibt es Abzug in der manuellen Bewertung (siehe weiter unten).

## Datensatz

Du erhältst eine ZIP-Datei `train_data.zip` mit Trainingsdaten: Paare aus `.txt`-Anzeige und `.json`-Label im obigen Schema. Auf den Testdaten (die du nicht siehst) wird dein Modell am Ende automatisch bewertet.

## Abgabe

Du gibst eine **ZIP-Datei** ab, die folgendes enthält:

- `notebook.ipynb` – dieses Notebook, mit deinen Implementierungen
- `model/` – ein Ordner mit deinem trainierten Modell (Tokenizer, Modellgewichte, ggf. weitere Artefakte wie Mapping-Dictionaries)

Ich werde **nur die Zellen bis einschließlich der Auswertung** ausführen. Dein Trainings-Code wird **nicht** erneut ausgeführt, sondern nur gesichtet. Daher ist es sehr wichtig, dass dein Modell in der `predict()`-Funktion korrekt aus dem `model/`-Ordner geladen wird.

## Bewertung

Insgesamt **30 Punkte**:

### 10 Punkte automatisch

Berechnet über `evaluate()` als Macro-F1 über alle 7 Felder.
- Macro-F1 < 0.3 → 0 Punkte
- Macro-F1 ≥ 0.85 → 10 Punkte
- dazwischen linear

### 20 Punkte manuell

In fünf Bereichen à 4 Punkten:

| Bereich | Was wird bewertet? |
|---|---|
| **Methoden- und Modellwahl** | Passt das Modell zur Aufgabe? Begründet? Alternativen erwogen? |
| **Data Understanding & Preprocessing** | Sinnvoll exploriert? Preprocessing zur Modellwahl passend? Sauberer Split? |
| **Trainings- und Inferenz-Pipeline** | Reproduzierbar? Hyperparameter begründet? Code lesbar? |
| **Evaluation & Fehleranalyse** | Eigene Evaluation jenseits von `evaluate()`? F1 pro Feld? Schwächen erkannt? |
| **Reflexion & Begründung** | Werden Entscheidungen in Markdown-Zellen erklärt? Wissenschaftlicher Stil? |
---

## Vorbereitungen

Lasse die folgenden Zellen unverändert. Sie installieren benötigte Pakete, definieren das Zielschema und entpacken die Trainingsdaten.

### Installations

In [1]:
!pip install -U \
numpy==1.26.4 \
    pandas==2.2.2 \
    pyarrow==15.0.2 \
    datasets==2.20.0 \
    huggingface-hub==0.23.2 \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    accelerate==0.33.0 \
    sentencepiece==0.2.1 \
    fsspec==2024.2.0 \
    fsspec==2024.2.0 \
    peft==0.11.1 \
    matplotlib \
    scikit-learn \
    pydantic \
    hf_transfer

!pip uninstall -y torch torchvision torchaudio

!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu121
!pip install -q lm-format-enforcer

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached pyarrow-15.0.2-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached datasets-2.20.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-0.23.2-py3-none-any.whl.metadata (12 kB)
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
  Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached accelerate-0.33.0-py3-none-any.whl.metadata (18 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached fsspec-2024.2.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached peft-0.11.1-py3-none-any.whl.metadata (13 kB)
  Using cached matplotlib-3.11.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86

In [2]:
from enum import StrEnum
from typing import Optional
from pydantic import BaseModel, Field


class ExperienceLevel(StrEnum):
    ENTRY = "entry"
    JUNIOR = "junior"
    MID = "mid"
    SENIOR = "senior"
    LEAD = "lead"
    EXECUTIVE = "executive"
    UNKNOWN = "unknown"


class SalaryRange(BaseModel):
    min: Optional[int] = None
    max: Optional[int] = None
    currency: Optional[str] = None


class JobPosting(BaseModel):
    job_title: str = ""
    company: str = ""
    experience_level: ExperienceLevel = ExperienceLevel.UNKNOWN
    required_skills: list[str] = Field(default_factory=list)
    education_requirements: str = ""
    salary_range: SalaryRange = Field(default_factory=SalaryRange)
    compensation_benefits: list[str] = Field(default_factory=list)

In [3]:
import zipfile
from pathlib import Path

TRAIN_ZIP = Path("train_data.zip")
TRAIN_DIR = Path("train")

if not TRAIN_DIR.exists():
    with zipfile.ZipFile(TRAIN_ZIP) as zf:
        zf.extractall(".")
print(f"Train: {len(list(TRAIN_DIR.glob('*.txt')))} Anzeigen")

FileNotFoundError: [Errno 2] No such file or directory: 'train_data.zip'

## Modell laden und Inferenz

In diesem Abschnitt wird dein **trainiertes Modell aus dem `model/`-Ordner geladen** und die `predict()`-Funktion bereitgestellt. Dieser Abschnitt wird bei der Korrektur ausgeführt – die Trainings-Sektion (weiter unten) hingegen nicht.

Passe die folgenden Zellen so an, dass:
1. dein Modell und alle benötigten Artefakte aus dem lokalen `model/`-Ordner geladen werden,
2. die `predict()`-Funktion ein gültiges `JobPosting`-Objekt zurückgibt.

**Verändere die Signatur und den Rückgabetyp von `predict()` nicht.**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import PeftModel
# Lade hier dein trainiertes Modell und alle benötigten Artefakte aus dem `model/`-Ordner.
# Beispiel für ein Hugging-Face-Modell:
#
#   from transformers import AutoTokenizer, AutoModelForTokenClassification
#   MODEL_DIR = Path("model")
#   tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
#   model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)

# ...
# model = AutoModelForCausalLM.from_pretrained("workspace/model",
#                                              device_map="cuda",
#                                              torch_dtype="auto")
# tokenizer = AutoTokenizer.from_pretrained("workspace/model")
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cuda", torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)
peft_model = PeftModel.from_pretrained(model, "workspace/lora_adapter")


In [39]:
from pydantic import ValidationError

def predict(text: str) -> JobPosting:
    parser = JsonSchemaParser(JobPosting.model_json_schema())
    prefix_fn = build_transformers_prefix_allowed_tokens_fn(tokenizer, parser)
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": text
        }
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(peft_model.device)

    output_ids = peft_model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=None,
        top_p=None,
        do_sample=False,
        prefix_allowed_tokens_fn=prefix_fn,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    output_ids = output_ids[0].cpu().tolist()

    job_posting_json = tokenizer.decode(output_ids[inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(job_posting_json)

    try:
        job_obj = JobPosting.model_validate_json(job_posting_json)
    except ValidationError:
        print("ValidationError: Das generierte JSON entspricht nicht dem Schema.")
        job_obj = JobPosting(
            job_title="",
            company="",
            experience_level=ExperienceLevel.UNKNOWN,
            required_skills=[],
            education_requirements="",
            salary_range=SalaryRange(),
            compensation_benefits=[],
        )

    return job_obj

## Auswertung

**Lasse die `evaluate()`-Funktion unverändert.** Sie bewertet dein Modell und berechnet die automatischen Punkte (max. 10).

Während der Entwicklung kannst du `EVAL_DIR` in der nächsten Zelle z.B. auf `Path("train")` setzen, um die Evaluation auf den Trainingsdaten laufen zu lassen und ein erstes Gefühl für die Modellgüte zu bekommen. Die Performance auf den Trainingsdaten überschätzt allerdings, wie gut dein Modell auf neuen Daten generalisiert, ein eigener Train/Val-Split ist methodisch sauberer.

In [40]:
import json
import math
import time
from pathlib import Path


def _norm_text(s: str) -> str:
    """Normalisiert Strings für robusten Vergleich (lowercase, getrimmt, Whitespace zusammengezogen)."""
    return " ".join((s or "").lower().split())


def _f1(tp: int, fp: int, fn: int) -> float:
    if tp == 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    return 2 * precision * recall / (precision + recall)


def _score_string_field(pred: str, true: str) -> tuple[int, int, int]:
    """Für String-Felder: token-set-basierter F1. Liefert (tp, fp, fn)."""
    p_tokens = set(_norm_text(pred).split())
    t_tokens = set(_norm_text(true).split())
    if not p_tokens and not t_tokens:
        return (1, 0, 0)  # beide leer -> perfekter Match
    tp = len(p_tokens & t_tokens)
    fp = len(p_tokens - t_tokens)
    fn = len(t_tokens - p_tokens)
    return (tp, fp, fn)


def _score_enum_field(pred: str, true: str) -> tuple[int, int, int]:
    """Für Enum-Felder: exakter Match."""
    if pred == true:
        return (1, 0, 0)
    return (0, 1, 1)


def _score_list_field(pred: list[str], true: list[str]) -> tuple[int, int, int]:
    """Für Listen-Felder: Set-F1 mit normalisierten Items."""
    p_set = {_norm_text(x) for x in pred if x}
    t_set = {_norm_text(x) for x in true if x}
    if not p_set and not t_set:
        return (1, 0, 0)
    tp = len(p_set & t_set)
    fp = len(p_set - t_set)
    fn = len(t_set - p_set)
    return (tp, fp, fn)


def _score_salary_field(pred: dict, true: dict) -> tuple[int, int, int]:
    """Für salary_range: F1 über die drei Subfelder. Toleranz für min/max: 10%."""
    tp = fp = fn = 0
    for key in ("min", "max"):
        p_val, t_val = pred.get(key), true.get(key)
        if p_val is None and t_val is None:
            tp += 1
        elif p_val is None and t_val is not None:
            fn += 1
        elif p_val is not None and t_val is None:
            fp += 1
        else:
            # 10% Toleranz
            if abs(p_val - t_val) <= max(1000, 0.1 * t_val):
                tp += 1
            else:
                fp += 1
                fn += 1
    # Currency: exakt
    p_cur, t_cur = pred.get("currency"), true.get("currency")
    if p_cur is None and t_cur is None:
        tp += 1
    elif (p_cur or "").upper() == (t_cur or "").upper() and p_cur:
        tp += 1
    else:
        if p_cur is not None:
            fp += 1
        if t_cur is not None:
            fn += 1
    return (tp, fp, fn)


FIELD_SCORERS = {
    "job_title": _score_string_field,
    "company": _score_string_field,
    "experience_level": _score_enum_field,
    "required_skills": _score_list_field,
    "education_requirements": _score_string_field,
    "salary_range": _score_salary_field,
    "compensation_benefits": _score_list_field,
}

INFERENCE_TIME_LIMIT_SECONDS = 20 * 60  # 20 Minuten


def evaluate(test_folder: Path) -> int:
    """Bewertet das Modell auf dem Testset und gibt die automatischen Punkte (0-10) zurück."""
    test_folder = Path(test_folder)
    txt_files = sorted(test_folder.glob("*.txt"))
    if not txt_files:
        raise FileNotFoundError(f"Keine .txt-Dateien in {test_folder} gefunden.")

    field_counts = {f: [0, 0, 0] for f in FIELD_SCORERS}

    start_time = time.time()
    for txt_file in txt_files:
        json_file = txt_file.with_suffix(".json")
        if not json_file.exists():
            print(f"WARNUNG: Kein Label für {txt_file.name}, übersprungen.")
            continue

        text = txt_file.read_text(encoding="utf-8")
        true = json.loads(json_file.read_text(encoding="utf-8"))
        pred = predict(text).model_dump()

        for field, scorer in FIELD_SCORERS.items():
            tp, fp, fn = scorer(pred[field], true[field])
            field_counts[field][0] += tp
            field_counts[field][1] += fp
            field_counts[field][2] += fn
    inference_time = time.time() - start_time

    print("\n=== F1 pro Feld ===")
    f1_per_field = {}
    for field, (tp, fp, fn) in field_counts.items():
        f1 = _f1(tp, fp, fn)
        f1_per_field[field] = f1
        print(f"  {field:25s}  F1={f1:.3f}  (TP={tp}, FP={fp}, FN={fn})")

    macro_f1 = sum(f1_per_field.values()) / len(f1_per_field)
    print(f"\nMacro-F1: {macro_f1:.3f}")

    # Punkteberechnung: <0.3 -> 0 Punkte, >=0.85 -> 10 Punkte, dazwischen linear
    if macro_f1 < 0.3:
        points = 0
    else:
        points = min(10, math.ceil((macro_f1 - 0.3) * 18))
    print(f"Automatische Punkte: {points} / 10")

    # Inferenz-Laufzeit
    print(f"\nInferenzzeit: {inference_time:.1f}s "
          f"({inference_time/len(txt_files):.2f}s pro Anzeige)")
    if inference_time > INFERENCE_TIME_LIMIT_SECONDS:
        print(f"WARNUNG: Inferenz hat das 20-Minuten-Limit überschritten "
              f"({inference_time/60:.1f} Min).")

    return points

In [ ]:
# TEST_ZIP = Path("test_data.zip")
# TEST_DIR = Path("test")
# if not TEST_DIR.exists() and TEST_ZIP.exists():
#     with zipfile.ZipFile(TEST_ZIP) as zf:
#         zf.extractall(".")
# print(f"Test: {len(list(TEST_DIR.glob('*.txt')))} Anzeigen")

In [41]:
# Während der Entwicklung kannst du EVAL_DIR z.B. auf Path("train") setzen,
# um die Evaluation auf den Trainingsdaten auszuprobieren.
# Für die offizielle Bewertung wird hier Path("test") gesetzt.
EVAL_DIR = Path("workspace/test")

evaluate(EVAL_DIR)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


{"job_title": "Senior Sales Marketing Specialist Programmatic", "company": "The Trade Desk", "experience_level": "senior", "required_skills": ["passionate and knowledgeable about the ad tech ecosystem and the programmatic industry", "years of level relevant experience in sales development, media planning, digital strategy, sales, corporate strategy or consulting", "experienced working across several channels (CTV and DOOH)"], "education_requirements": "four year college degree level education"}
{
  "job_title": "Construction Project Manager",
  "company": "Tractor Supply Company (TSC) - Northeast Market",
  "required_skills": [
    "Experience: Years of related business experience",
    "Education: Bachelors degree from an accredited college or university in Construction Management or related field is preferred, any suitable combination of education and experience will be considered"
  ],
  "experience_level": "entry",
  "salary_range": {
    "min": null,
    "max": null
  },
  "educat

3

---

# Trainings-Dokumentation

**Hinweis:** Der folgende Abschnitt dokumentiert dein Vorgehen beim Training. Er wird bei der Korrektur **nicht** ausgeführt, sondern nur gesichtet (das fertige Modell liegt ja bereits im `model/`-Ordner und wird oben geladen).

Dieser Abschnitt fließt in die **manuelle Bewertung** ein. Achte auf:
- Nachvollziehbare Darstellung deiner Entscheidungen (in Markdown-Zellen)
- Reproduzierbarer Code (falls jemand das Training nachvollziehen will)
- Sinnvolle Visualisierungen und Statistiken

## Data Understanding

Untersuche die bereitgestellten Daten. Sinnvolle Fragen können sein:
- Wie lang sind die Anzeigen typischerweise (in Tokens / Zeichen)?
- Wie ist die Verteilung der `experience_level`-Klassen?
- Wie viele Anzeigen enthalten überhaupt eine Gehaltsangabe?
- Welche Skills tauchen am häufigsten auf?

Diese Einsichten helfen dir, das richtige Modell und passende Preprocessing-Schritte zu wählen.

In [ ]:
# ...

## Data Preprocessing

Bereite die Daten so vor, dass sie zu deinem gewählten Modellansatz passen.
Mögliche Schritte: Tokenisierung, Konvertierung in Seq2Seq-Format (z.B. `text -> JSON-String`), Erzeugung von BIO-Tags für NER, Train/Val-Split, etc.

In [28]:
from pathlib import Path
import json
import pandas as pd
from sklearn.model_selection import train_test_split

train_folder = Path("./workspace/train")
txt_files = sorted(train_folder.glob("*.txt"))

if not txt_files:
    raise FileNotFoundError(f"Keine .txt-Dateien in {train_folder} gefunden.")

rows = []

for txt_file in txt_files:
    json_file = txt_file.with_suffix(".json")

    if not json_file.exists():
        print(f"WARNUNG: Kein Label für {txt_file.name}, übersprungen.")
        continue

    text = txt_file.read_text(encoding="utf-8")
    labels = json.loads(json_file.read_text(encoding="utf-8"))

    row = {
        "filename": txt_file.stem,
        "text": text,
        "label": json.dumps(labels, ensure_ascii=False)
    }

    rows.append(row)

df = pd.DataFrame(rows)

print(df.head())
print(f"Anzahl Dokumente: {len(df)}")

   filename                                               text  \
0  job_0000  description\n\nabout this role\n\noverview\n\n...   
1  job_0001  due to covid in effort to embrace social dista...   
2  job_0002  purpose\nthe warehouse supervisor is responsib...   
3  job_0003  description hughes private capital in business...   
4  job_0004  description\n\ndevelops and implements effecti...   

                                               label  
0  {"job_title": "Integrated Marketing, VP (Retir...  
1  {"job_title": "Customer Service Representative...  
2  {"job_title": "Warehouse Supervisor", "company...  
3  {"job_title": "Software Engineer", "company": ...  
4  {"job_title": "Clinical Educator Part Time", "...  
Anzahl Dokumente: 559


In [29]:
train_val_df, eval_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(f"Train: {len(train_df)}")
print(f"Val: {len(eval_df)}")

Train: 448
Val: 112


In [30]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_ds = Dataset.from_pandas(eval_df.reset_index(drop=True))

In [6]:
import json
from collections import Counter

bad = 0
starts = Counter()
examples = []

for x in train_ds.select(range(min(500, len(train_ds)))):
    y = x["label"]
    s = y.strip() if isinstance(y, str) else str(y)
    starts[s[:1]] += 1
    try:
        json.loads(s)
    except Exception:
        bad += 1
        if len(examples) < 5:
            examples.append(s[:300])

print("invalid json:", bad)
print("start chars:", starts.most_common(10))
print("bad examples:")
for e in examples:
    print("---", e)

invalid json: 0
start chars: [('{', 448)]
bad examples:


In [7]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

In [31]:
system_prompt = "Extract structured information from job advertisements and answer only with valid JSON."
max_length = 2048
max_target_tokens = 256


def preprocess_inputs(samples):
    input_ids_list = []
    attention_masks = []
    labels_list = []

    for text, target in zip(samples["text"], samples["label"]):
        prompt_msgs = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text},
        ]

        prompt_ids = tokenizer.apply_chat_template(
            prompt_msgs,
            tokenize=True,
            add_generation_prompt=True,
        )

        target_ids = tokenizer(target, add_special_tokens=False)["input_ids"]
        target_ids = target_ids[:max_target_tokens] + [tokenizer.eos_token_id]

        available_for_prompt = max_length - len(target_ids)
        if available_for_prompt <= 0:
            # extrem langes Json
            continue

        if len(prompt_ids) > available_for_prompt:
            sys_prompt_ids = tokenizer.apply_chat_template(
                [{"role": "system", "content": system_prompt},
                 {"role": "user", "content": ""}],
                tokenize=True,
                add_generation_prompt=True,
            )
            system_overhead = len(sys_prompt_ids)

            user_budget = max(16, available_for_prompt - system_overhead)
            user_ids = tokenizer(text, add_special_tokens=False)["input_ids"][:user_budget]
            text_cut = tokenizer.decode(user_ids, skip_special_tokens=True)

            prompt_msgs = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": text_cut},
            ]
            prompt_ids = tokenizer.apply_chat_template(
                prompt_msgs,
                tokenize=True,
                add_generation_prompt=True,
            )

            # falls wegen Template-Overhead noch minimal zu lang
            prompt_ids = prompt_ids[:available_for_prompt]

        input_ids = prompt_ids + target_ids
        input_ids = input_ids[:max_length]

        labels = [-100] * len(prompt_ids) + target_ids
        labels = labels[:max_length]

        attention_mask = [1] * len(input_ids)

        pad_len = max_length - len(input_ids)
        input_ids += [tokenizer.pad_token_id] * pad_len
        attention_mask += [0] * pad_len
        labels += [-100] * pad_len

        input_ids_list.append(input_ids)
        attention_masks.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks,
        "labels": labels_list,
    }

In [32]:
tokenized_train_ds = train_ds.map(preprocess_inputs, batched=True, remove_columns=train_ds.column_names, keep_in_memory=False, )
print(tokenizer.decode(tokenized_train_ds[0]["input_ids"]))

Map:   0%|          | 0/448 [00:00<?, ? examples/s]

<|im_start|>system
Extract structured information from job advertisements and answer only with valid JSON.<|im_end|>
<|im_start|>user
job description
salary

who we are

sjr is an awardwinning content consultancy we believe in the power of transformative content and have an unparalleled group of people with uncommon talents who do content marketing differently we combine journalistic rigor with creative intelligence then amplify with technology to sharpen the conversation decode complexity transfer knowledge and build trust

we hire passionate innovators disruptors and content marketers to join us on our mission to create bestinclass brand storytelling we strategize and create for the worlds best and biggest brands because they have such an impact on society we have a big impact too

who we are looking for

we are currently seeking a senior art director to support and uphold the visual aesthetics of our creative efforts the right candidate is a multidisciplinary creative leader further

In [33]:
tokenized_eval_ds = eval_ds.map(preprocess_inputs, batched=True, remove_columns=eval_ds.column_names, keep_in_memory=False, )

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

In [11]:
import numpy as np

# 1) Verteilung der supervised Tokens im bereits tokenisierten Train-Datensatz
supervised_counts = []
seq_lens = []
ratios = []

for ex in tokenized_train_ds:
    lbl = ex["labels"]
    n_sup = sum(t != -100 for t in lbl)
    n_len = len(lbl)
    supervised_counts.append(n_sup)
    seq_lens.append(n_len)
    ratios.append(n_sup / n_len if n_len else 0.0)

print("=== Supervised token stats (train) ===")
print("samples:", len(supervised_counts))
print("min / p25 / median / p75 / max:",
      np.min(supervised_counts),
      np.percentile(supervised_counts, 25),
      np.median(supervised_counts),
      np.percentile(supervised_counts, 75),
      np.max(supervised_counts))
print("mean supervised tokens:", float(np.mean(supervised_counts)))
print("mean supervised ratio:", float(np.mean(ratios)))
print("samples with 0 supervised tokens:", sum(x == 0 for x in supervised_counts))

# 2) Spot-check: Ist das Ziel-JSON im tokenisierten Beispiel überhaupt noch "drin"?
#    (Vergleich zwischen erwarteter Label-Tokenlänge und tatsächlich supervised Tokens)
N = min(10, len(train_ds))
print("\n=== Spot check first", N, "raw samples ===")
for i in range(N):
    raw = train_ds[i]
    tok = tokenized_train_ds[i]

    # erwartete Länge des Assistant-Contents (ohne spezielles Masking)
    target_ids = tokenizer(raw["label"], add_special_tokens=False)["input_ids"]
    expected_target_len = len(target_ids)

    actual_supervised = sum(t != -100 for t in tok["labels"])

    print(f"[{i}] expected_target_len={expected_target_len:4d} | actual_supervised={actual_supervised:4d}")

=== Supervised token stats (train) ===
samples: 448
min / p25 / median / p75 / max: 61 104.75 125.0 148.0 257
mean supervised tokens: 129.08705357142858
mean supervised ratio: 0.06303078787667411
samples with 0 supervised tokens: 0

=== Spot check first 10 raw samples ===
[0] expected_target_len= 106 | actual_supervised= 107
[1] expected_target_len= 123 | actual_supervised= 124
[2] expected_target_len= 128 | actual_supervised= 129
[3] expected_target_len= 132 | actual_supervised= 133
[4] expected_target_len= 108 | actual_supervised= 109
[5] expected_target_len=  92 | actual_supervised=  93
[6] expected_target_len= 118 | actual_supervised= 119
[7] expected_target_len= 110 | actual_supervised= 111
[8] expected_target_len= 101 | actual_supervised= 102
[9] expected_target_len= 147 | actual_supervised= 148


## Modelldefinition und Training

Definiere und trainiere dein Modell. Begründe in einer Markdown-Zelle kurz, warum du dich für diesen Ansatz entschieden hast.

**Wichtig:** Speichere dein trainiertes Modell am Ende dieses Abschnitts in den Ordner `model/`, damit die `predict()`-Funktion ganz oben es laden kann.

Beispiel für ein Hugging-Face-Modell:
```python
model.save_pretrained("model")
tokenizer.save_pretrained("model")
```

In [34]:
from transformers import AutoModelForSeq2SeqLM, AutoModelForCausalLM
import gc, torch

if 'model' in locals():
    del model
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
model = model.to("cuda")

model.config.use_cache = False
model.gradient_checkpointing_enable()

In [35]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
)

# get_peft_model() wickelt das Original-Modell ein:
# - Friert alle Original-Parameter ein (requires_grad = False)
# - Fügt LoRA-Matrizen für die target_modules hinzu (requires_grad = True)
peft_model = get_peft_model(model, lora_config)

peft_model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [36]:
from transformers import TrainingArguments, Trainer, default_data_collator

training_args = TrainingArguments(
    output_dir="workspace/stellenanzeigen_model",
    learning_rate=5e-6,
    weight_decay=0.01,
    num_train_epochs=3,
    report_to="none",
    max_grad_norm=1.0,
    warmup_ratio=0.03,
    logging_steps=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    bf16=True,
    fp16=False,
    optim="adamw_torch"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_eval_ds,
    data_collator=default_data_collator,
)

# Training starten.
trainer.train()

Step,Training Loss
5,1.445400
10,1.526800
15,1.415000
20,1.496100
25,1.323600
30,1.355400
35,1.325200
40,1.370700
45,1.339300
50,1.272600


TrainOutput(global_step=168, training_loss=1.2578558524449666, metrics={'train_runtime': 1236.9842, 'train_samples_per_second': 1.087, 'train_steps_per_second': 0.136, 'total_flos': 2.1792842972135424e+16, 'train_loss': 1.2578558524449666, 'epoch': 3.0})

In [37]:
logs = trainer.state.log_history
train_losses = [x["loss"] for x in logs if "loss" in x]
print("first losses:", train_losses[:5])
print("last losses:", train_losses[-5:])

first losses: [1.4454, 1.5268, 1.415, 1.4961, 1.3236]
last losses: [1.1579, 1.0906, 1.1696, 1.0983, 1.179]


In [38]:
peft_model.save_pretrained("workspace/lora_adapter")

In [17]:
model.save_pretrained("workspace/model")
tokenizer.save_pretrained("workspace/model")

('workspace/model/tokenizer_config.json',
 'workspace/model/special_tokens_map.json',
 'workspace/model/vocab.json',
 'workspace/model/merges.txt',
 'workspace/model/added_tokens.json',
 'workspace/model/tokenizer.json')

## Evaluation und Fehleranalyse

Über die `evaluate()`-Funktion hinaus: Wo macht dein Modell typische Fehler? Welche Felder sind am schwierigsten zu extrahieren und warum? Was wären sinnvolle nächste Schritte, um die Performance zu verbessern?

Erstelle hier eine kurze Fehleranalyse (z.B. F1 pro Feld auf einem Validierungs-Split, Beispiele für Fehler, Verwechslungsmatrix für `experience_level`).

In [18]:
peft_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear(
                (base_layer): Linear(in_features=153

In [21]:
from lmformatenforcer import JsonSchemaParser
from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn

def extract_with_json_template(job, max_new_tokens=256, **gen_kwargs):
    parser = JsonSchemaParser(JobPosting.model_json_schema())
    prefix_fn = build_transformers_prefix_allowed_tokens_fn(tokenizer, parser)
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": job
        }
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(peft_model.device)

    output_ids = peft_model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=None,
        top_p=None,
        do_sample=False,
        prefix_allowed_tokens_fn=prefix_fn,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    # Beispiel: nach einem generate-Aufruf (funktion extract_with_model / extract_with_json_template)
    output_ids = output_ids[0].cpu().tolist()  # list of ints

    # ganze Liste der IDs und ersten 200 IDs anzeigen
    print("output ids (first 200):", output_ids[:200])

    # token strings für die ersten 100 IDs
    print("tokens (first 100):", tokenizer.convert_ids_to_tokens(output_ids[:100]))

    # dekodierte String-Ausgabe (ganze Ausgabe)
    print("decoded:", tokenizer.decode(output_ids, skip_special_tokens=False))
    job_posting_json = tokenizer.decode(output_ids[inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(job_posting_json)

    job_obj = JobPosting.model_validate_json(job_posting_json)
    return job_obj

In [19]:

def extract(job, max_new_tokens=256, **gen_kwargs):
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": job
        }
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(peft_model.device)

    output_ids = peft_model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=None,
        top_p=None,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    # Beispiel: nach einem generate-Aufruf (funktion extract_with_model / extract_with_json_template)
    output_ids = output_ids[0].cpu().tolist()  # list of ints

    # ganze Liste der IDs und ersten 200 IDs anzeigen
    print("output ids (first 200):", output_ids[:200])

    # token strings für die ersten 100 IDs
    print("tokens (first 100):", tokenizer.convert_ids_to_tokens(output_ids[:100]))

    # dekodierte String-Ausgabe (ganze Ausgabe)
    print("decoded:", tokenizer.decode(output_ids, skip_special_tokens=False))
    job_posting_json = tokenizer.decode(output_ids[inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    return job_posting_json

In [22]:
test_head = eval_ds.select(range(10))
inputs = test_head["text"]
for i, sample in enumerate(inputs):
    print(f"Text:     {test_head[i]['text']}")
    print(f"JSON:     {extract_with_json_template(sample)}")
    print("-" * 100)

Text:     sr sap basis administrator

provide administration and technical support for all of j crews sap systems this job covers all aspects of sap basis administration including system installations and upgrades problem analysis and resolution database management client copies system refreshes sap instance configuration and performance tuning for a complex system landscape

duties and responsibilities
 working at a technical level with an understanding of sap its underlying database operating system and hardware platform
 interact with development teams configuration teams various technical support teams and business stakeholders to optimize and maintain the overall sap reliability availability and performance
 the position has a  oncall commitment and availability for evening and weekend working is required
 provide installation configuration integration upgrade and testing services for new projects and ongoing maintenance for sap environments in a vm environment
 analyze diagnose a

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


output ids (first 200): [151644, 8948, 198, 28959, 32930, 1995, 504, 2618, 42649, 323, 4226, 1172, 448, 2697, 4718, 13, 151645, 198, 151644, 872, 198, 15094, 34635, 8037, 28093, 271, 61456, 8567, 323, 10916, 1824, 369, 678, 315, 502, 42060, 34635, 5942, 419, 2618, 14521, 678, 13566, 315, 34635, 8037, 8567, 2670, 1849, 44118, 323, 31614, 3491, 6358, 323, 10935, 4625, 6240, 2943, 10993, 1849, 10408, 288, 34635, 2867, 6546, 323, 5068, 41338, 369, 264, 6351, 1849, 18414, 271, 67, 332, 550, 323, 27323, 198, 3238, 518, 264, 10916, 2188, 448, 458, 8660, 315, 34635, 1181, 16533, 4625, 10350, 1849, 323, 11773, 5339, 198, 16282, 448, 4401, 7263, 6546, 7263, 5257, 10916, 1824, 7263, 323, 2562, 38110, 311, 29436, 323, 10306, 279, 8084, 34635, 30538, 18048, 323, 5068, 198, 279, 2309, 702, 264, 220, 389, 6659, 15155, 323, 18048, 369, 11458, 323, 9001, 3238, 374, 2567, 198, 3410, 13713, 6546, 17590, 13910, 323, 7497, 3516, 369, 501, 7079, 323, 14195, 13404, 369, 34635, 21737, 304, 264, 10995, 4573, 1

ValidationError: 1 validation error for JobPosting
  Invalid JSON: EOF while parsing a string at line 26 column 51 [type=json_invalid, input_value='{\n  "job_title": "Comme...mbursement for Personal', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid

In [20]:
test_head = eval_ds.select(range(10))
inputs = test_head["text"]
for i, sample in enumerate(inputs):
    print(f"Text:     {test_head[i]['text']}")
    print(f"JSON:     {extract(sample)}")
    print("-" * 100)

Text:     sr sap basis administrator

provide administration and technical support for all of j crews sap systems this job covers all aspects of sap basis administration including system installations and upgrades problem analysis and resolution database management client copies system refreshes sap instance configuration and performance tuning for a complex system landscape

duties and responsibilities
 working at a technical level with an understanding of sap its underlying database operating system and hardware platform
 interact with development teams configuration teams various technical support teams and business stakeholders to optimize and maintain the overall sap reliability availability and performance
 the position has a  oncall commitment and availability for evening and weekend working is required
 provide installation configuration integration upgrade and testing services for new projects and ongoing maintenance for sap environments in a vm environment
 analyze diagnose a

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


output ids (first 200): [151644, 8948, 198, 28959, 32930, 1995, 504, 2618, 42649, 323, 4226, 1172, 448, 2697, 4718, 13, 151645, 198, 151644, 872, 198, 15094, 34635, 8037, 28093, 271, 61456, 8567, 323, 10916, 1824, 369, 678, 315, 502, 42060, 34635, 5942, 419, 2618, 14521, 678, 13566, 315, 34635, 8037, 8567, 2670, 1849, 44118, 323, 31614, 3491, 6358, 323, 10935, 4625, 6240, 2943, 10993, 1849, 10408, 288, 34635, 2867, 6546, 323, 5068, 41338, 369, 264, 6351, 1849, 18414, 271, 67, 332, 550, 323, 27323, 198, 3238, 518, 264, 10916, 2188, 448, 458, 8660, 315, 34635, 1181, 16533, 4625, 10350, 1849, 323, 11773, 5339, 198, 16282, 448, 4401, 7263, 6546, 7263, 5257, 10916, 1824, 7263, 323, 2562, 38110, 311, 29436, 323, 10306, 279, 8084, 34635, 30538, 18048, 323, 5068, 198, 279, 2309, 702, 264, 220, 389, 6659, 15155, 323, 18048, 369, 11458, 323, 9001, 3238, 374, 2567, 198, 3410, 13713, 6546, 17590, 13910, 323, 7497, 3516, 369, 501, 7079, 323, 14195, 13404, 369, 34635, 21737, 304, 264, 10995, 4573, 1

KeyboardInterrupt: 

In [30]:
print("pad_token_id:", tokenizer.pad_token_id, "->", tokenizer.convert_ids_to_tokens([tokenizer.pad_token_id]) if tokenizer.pad_token_id is not None else None)
print("bos_token_id:", tokenizer.bos_token_id, "->", tokenizer.convert_ids_to_tokens([tokenizer.bos_token_id]) if tokenizer.bos_token_id is not None else None)
print("eos_token_id:", tokenizer.eos_token_id, "->", tokenizer.convert_ids_to_tokens([tokenizer.eos_token_id]) if tokenizer.eos_token_id is not None else None)
print("all special tokens map:", tokenizer.special_tokens_map)

pad_token_id: 151643 -> ['<|endoftext|>']
bos_token_id: None -> None
eos_token_id: 151645 -> ['<|im_end|>']
all special tokens map: {'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}


In [ ]:
# ...
